<a href="https://colab.research.google.com/github/Usman-Saleem-250547/retail-ai/blob/main/Copy_of_mallAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install ultralytics fastapi uvicorn nest-asyncio pyngrok
!apt-get install -y libgl1-mesa-glx libglib2.0-0
print("✅ Done")

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libglib2.0-0 is already the newest version (2.72.4-0ubuntu2.9).
libgl1-mesa-glx is already the newest version (23.0.4-0ubuntu1~22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 3 not upgraded.
✅ Done


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('/content/static', exist_ok=True)
os.makedirs('/content/drive/MyDrive/MartAI', exist_ok=True)
print("✅ Ready")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Ready


### 1. Updating `tracker.py`

We need to modify `tracker.py` to save the processed frames as an image file that the web server can then pick up and serve. This means removing the `cv2.imshow` call and instead saving the frame to `static/current_frame.jpg`.


In [ ]:
tracker_code = '''import cv2
from ultralytics import YOLO
import time, sqlite3, os
from datetime import datetime

CAMERA_URL = "/content/supermarket_video.mp4"
DB_FILE = "/content/drive/MyDrive/MartAI/store_data.db"
LOG_EVERY = 5
FRAME_SAVE_PATH = "/content/static/current_frame.jpg"

ZONES = [
    ("Entrance", 0,   0,   320, 240),
    ("Aisle_1",  320, 0,   640, 240),
    ("Checkout", 0,   240, 320, 480),
    ("Aisle_2",  320, 240, 640, 480),
]

def init_db():
    conn = sqlite3.connect(DB_FILE)
    conn.execute("""CREATE TABLE IF NOT EXISTS zone_logs (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        timestamp TEXT, zone_name TEXT, count INTEGER)""")
    conn.execute("""CREATE TABLE IF NOT EXISTS person_tracking (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        timestamp TEXT, track_id INTEGER, zone_name TEXT, event_type TEXT)""")
    conn.commit()
    conn.close()

def log_to_db(zone_counts):
    conn = sqlite3.connect(DB_FILE)
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    for zone, count in zone_counts.items():
        conn.execute("INSERT INTO zone_logs VALUES (NULL,?,?,?)", (ts, zone, count))
    conn.commit()
    conn.close()

def log_person_event(track_id, zone_name, event_type):
    conn = sqlite3.connect(DB_FILE)
    ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    conn.execute("INSERT INTO person_tracking VALUES (NULL,?,?,?,?)", (ts, track_id, zone_name, event_type))
    conn.commit()
    conn.close()

def get_zone(cx, cy):
    for name, x1, y1, x2, y2 in ZONES:
        if x1 <= cx <= x2 and y1 <= cy <= y2:
            return name
    return None

def draw_zones(frame):
    for name, x1, y1, x2, y2 in ZONES:
        cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 255), 2)
        cv2.putText(frame, name, (x1+6, y1+22), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,255), 2)

def main():
    init_db()
    model = YOLO("yolov8n.pt")
    cap = cv2.VideoCapture(CAMERA_URL)
    if not cap.isOpened():
        print("ERROR: Cannot open video:", CAMERA_URL)
        return
    print("Tracker running...")
    last_log = time.time()
    tracked = {}
    frame_n = 0
    while True:
        ret, frame = cap.read()
        if not ret:
            cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
            ret, frame = cap.read()
            if not ret:
                break
        frame = cv2.resize(frame, (640, 480))
        results = model.track(frame, persist=True, conf=0.5, tracker="bytetrack.yaml")
        current_ids = set()
        zone_counts = {z[0]: 0 for z in ZONES}
        if results[0].boxes.id is not None:
            boxes = results[0].boxes.xyxy.cpu().numpy()
            ids = results[0].boxes.id.cpu().numpy().astype(int)
            for box, tid in zip(boxes, ids):
                x1, y1, x2, y2 = map(int, box)
                cx, cy = (x1+x2)//2, (y1+y2)//2
                zone = get_zone(cx, cy)
                if zone:
                    current_ids.add(tid)
                    zone_counts[zone] += 1
                    cv2.rectangle(frame, (x1,y1), (x2,y2), (0,255,0), 2)
                    cv2.putText(frame, f"ID:{tid} {zone}", (x1, y1-10), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 2)
                    if tid not in tracked:
                        tracked[tid] = {"zone": zone, "last": datetime.now()}
                        log_person_event(tid, zone, "enter")
                    elif tracked[tid]["zone"] != zone:
                        log_person_event(tid, tracked[tid]["zone"], "exit")
                        tracked[tid] = {"zone": zone, "last": datetime.now()}
                        log_person_event(tid, zone, "enter")
                    else:
                        tracked[tid]["last"] = datetime.now()
        to_del = [t for t in tracked if t not in current_ids and
                  (datetime.now()-tracked[t]["last"]).total_seconds() > 2]
        for t in to_del:
            log_person_event(t, tracked[t]["zone"], "exit")
            del tracked[t]
        if time.time() - last_log >= LOG_EVERY:
            log_to_db(zone_counts)
            last_log = time.time()
        draw_zones(frame)
        frame_n += 1
        if frame_n % 2 == 0:
            cv2.imwrite(FRAME_SAVE_PATH, frame)
    cap.release()

if __name__ == "__main__":
    if not os.path.exists("/content/supermarket_video.mp4"):
        print("Downloading video...")
        os.system("wget -q -O /content/supermarket_video.mp4 https://www.learningcontainer.com/wp-content/uploads/2020/05/sample-mp4-file.mp4")
        print("Done.")
    main()
'''
with open('/content/tracker.py', 'w') as f:
    f.write(tracker_code)
print("✅ tracker.py written")

✅ tracker.py written


### 2. Setting up the Web Frontend

First, we need to create the static files for our web dashboard. These include:

*   `index.html`: The main page structure.
*   `style.css`: For styling the dashboard.
*   `script.js`: For fetching data from our API and updating the dashboard dynamically.


In [ ]:
# Create a directory for static files
!mkdir -p static

In [ ]:
import os

os.makedirs('/content/static', exist_ok=True)
print("✅ static folder created:", os.path.exists('/content/static'))

✅ static folder created: True


In [ ]:
%%writefile /content/static/index.html
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>Superstore AI Dashboard</title>
    <link rel="stylesheet" href="/static/style.css">
</head>
<body>
    <div class="container">
        <h1>🛒 Superstore AI Dashboard</h1>
        <div class="video-feed">
            <h2>Live Camera Feed</h2>
            <img id="liveFeed" src="/live_video_feed" width="640" height="480" alt="Feed">
        </div>
        <div class="dashboard-data">
            <div class="data-section"><h2>Zone Counts</h2><div id="zoneCounts">Loading...</div></div>
            <div class="data-section"><h2>Person Events</h2><div id="personEvents">Loading...</div></div>
        </div>
    </div>
    <script src="/static/script.js"></script>
</body>
</html>

Overwriting /content/static/index.html


In [ ]:
%%writefile /content/static/style.css
body { font-family: Arial, sans-serif; background: #f0f0f0; padding: 20px; }
.container { max-width: 960px; margin: auto; background: #fff; padding: 20px; border-radius: 10px; }
h1 { color: #0056b3; text-align: center; }
.video-feed { text-align: center; margin-bottom: 20px; }
#liveFeed { border: 3px solid #0056b3; border-radius: 6px; max-width: 100%; }
.dashboard-data { display: flex; gap: 20px; flex-wrap: wrap; margin-top: 20px; }
.data-section { flex: 1; min-width: 280px; background: #e9f5ff; border: 1px solid #b3d9ff; padding: 15px; border-radius: 8px; }
h2 { color: #004085; font-size: 1em; }
p { margin: 4px 0; font-size: 0.88em; }

Overwriting /content/static/style.css


In [ ]:
%%writefile /content/static/script.js
let lastId = 0;
const B = window.location.origin;
function updateFeed() {
    document.getElementById('liveFeed').src = B + '/live_video_feed?t=' + Date.now();
}
async function updateData() {
    try {
        const z = await (await fetch(B+'/zone_counts')).json();
        document.getElementById('zoneCounts').innerHTML =
            z.map(d=>`<p><b>${d.zone_name}:</b> ${d.count} people <small>(${d.timestamp})</small></p>`).join('') || '<p>No data yet</p>';
    } catch(e) {}
    try {
        const p = await (await fetch(B+`/person_events?last_id=${lastId}`)).json();
        if (p.length) {
            const div = document.getElementById('personEvents');
            p.forEach(d => {
                div.innerHTML = `<p><b>${d.timestamp}:</b> Person ${d.track_id} → ${d.event_type} ${d.zone_name}</p>` + div.innerHTML;
                lastId = Math.max(lastId, d.id);
            });
        }
    } catch(e) {}
}
setInterval(updateFeed, 2000);
setInterval(updateData, 3000);
updateFeed(); updateData();

Overwriting /content/static/script.js


### 3. Updating `api.py` to be a FastAPI Web Server

Now, we'll modify `api.py` to become a FastAPI web server that:

*   Serves the static HTML, CSS, and JavaScript files for the dashboard.
*   Provides an endpoint (`/live_video_feed`) to fetch the latest processed frame image from `tracker.py`.
*   Provides endpoints (`/zone_counts` and `/person_events`) to fetch the latest statistical data from the SQLite database.


In [ ]:
api_code = '''from fastapi import FastAPI, Response
from fastapi.staticfiles import StaticFiles
from fastapi.responses import FileResponse, JSONResponse
import sqlite3, os

app = FastAPI()
DB = "/content/drive/MyDrive/MartAI/store_data.db"
FRAME = "/content/static/current_frame.jpg"
STATIC = "/content/static"

app.mount("/static", StaticFiles(directory=STATIC), name="static")

@app.get("/")
def root(): return FileResponse(f"{STATIC}/index.html")

@app.get("/live_video_feed")
def video():
    if os.path.exists(FRAME):
        return FileResponse(FRAME, media_type="image/jpeg")
    return JSONResponse({"error": "no frame"}, 404)

@app.get("/zone_counts")
def zone_counts():
    conn = sqlite3.connect(DB)
    rows = conn.execute("""SELECT zl.zone_name, zl.count, zl.timestamp
        FROM zone_logs zl
        INNER JOIN (SELECT zone_name, MAX(id) as mid FROM zone_logs GROUP BY zone_name) m
        ON zl.zone_name=m.zone_name AND zl.id=m.mid""").fetchall()
    conn.close()
    return [{"zone_name":r[0],"count":r[1],"timestamp":r[2]} for r in rows]

@app.get("/person_events")
def person_events(last_id: int = 0):
    conn = sqlite3.connect(DB)
    rows = conn.execute("SELECT id,timestamp,track_id,zone_name,event_type FROM person_tracking WHERE id>? ORDER BY id DESC LIMIT 20", (last_id,)).fetchall()
    conn.close()
    return [{"id":r[0],"timestamp":r[1],"track_id":r[2],"zone_name":r[3],"event_type":r[4]} for r in rows]
'''
with open('/content/api.py', 'w') as f:
    f.write(api_code)
print("✅ api.py written")

✅ api.py written


In [ ]:
import os
if not os.path.exists("/content/supermarket_video.mp4"):
    !wget -q -O /content/supermarket_video.mp4 "https://www.learningcontainer.com/wp-content/uploads/2020/05/sample-mp4-file.mp4"
print("✅ Video ready:", os.path.exists("/content/supermarket_video.mp4"))

✅ Video ready: True


In [ ]:
import subprocess, time, os

os.system("pkill -f tracker.py")
time.sleep(2)

proc = subprocess.Popen(
    ["python", "/content/tracker.py"],
    stdout=open('/content/tracker_output.log', 'w'),
    stderr=subprocess.STDOUT
)
print(f"✅ Tracker started PID: {proc.pid}")
time.sleep(8)
os.system("cat /content/tracker_output.log")

✅ Tracker started PID: 17394


0

In [ ]:
import nest_asyncio, uvicorn, threading, importlib
from pyngrok import ngrok
import api

nest_asyncio.apply()
importlib.reload(api)
ngrok.kill()

# ⚠️ Put YOUR token here from ngrok.com (free signup)
ngrok.set_auth_token("3FDlUQ9doKwwbGfUYwrJ1FiLqBA_6XW2t2jx4uZZox6Dg3XKw")

url = ngrok.connect(8000).public_url
print("=" * 50)
print("✅ OPEN THIS URL:", url)
print("=" * 50)

threading.Thread(
    target=lambda: uvicorn.run(api.app, host="0.0.0.0", port=8000),
    daemon=True
).start()

# Keep alive without blocking
import time
while True:
    time.sleep(30)
    # Check tracker still running
    import subprocess
    r = subprocess.run(["pgrep", "-f", "tracker.py"], capture_output=True)
    if not r.stdout:
        print("⚠️ Tracker stopped! Check tracker_output.log")

✅ OPEN THIS URL: https://sensually-stinger-dwarf.ngrok-free.dev


INFO:     Started server process [6513]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
ERROR:    [Errno 98] error while attempting to bind on address ('0.0.0.0', 8000): address already in use
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.


INFO:     72.255.17.40:0 - "GET / HTTP/1.1" 200 OK
INFO:     72.255.17.40:0 - "GET /static/style.css HTTP/1.1" 200 OK
INFO:     72.255.17.40:0 - "GET /static/script.js HTTP/1.1" 200 OK
INFO:     72.255.17.40:0 - "GET /live_video_feed HTTP/1.1" 200 OK
INFO:     72.255.17.40:0 - "GET /live_video_feed?t=1781693978899 HTTP/1.1" 200 OK
INFO:     72.255.17.40:0 - "GET /zone_counts HTTP/1.1" 200 OK
INFO:     72.255.17.40:0 - "GET /person_events?last_id=0 HTTP/1.1" 200 OK
INFO:     72.255.17.40:0 - "GET /live_video_feed?t=1781693980910 HTTP/1.1" 200 OK
INFO:     72.255.17.40:0 - "GET /zone_counts HTTP/1.1" 200 OK
INFO:     72.255.17.40:0 - "GET /person_events?last_id=431 HTTP/1.1" 200 OK
INFO:     72.255.17.40:0 - "GET /live_video_feed?t=1781693982903 HTTP/1.1" 200 OK
INFO:     72.255.17.40:0 - "GET /zone_counts HTTP/1.1" 200 OK
INFO:     72.255.17.40:0 - "GET /live_video_feed?t=1781693984910 HTTP/1.1" 200 OK
INFO:     72.255.17.40:0 - "GET /person_events?last_id=431 HTTP/1.1" 200 OK
INFO:    

KeyboardInterrupt: 

**After running the cell above:**

1.  You should see a `Public URL for your FastAPI app: YOUR_NGROK_URL` printed.
2.  Click on this URL to open your live superstore dashboard in a new browser tab.
3.  The dashboard should now display the live video feed (from `current_frame.jpg` being updated by `tracker.py`) and real-time statistics from your database.

To stop the FastAPI server and ngrok tunnel, you can interrupt the execution of the Python cell above (e.g., by clicking the stop button next to the cell or Runtime -> Interrupt execution).